# A caplet and a cap: option value beyond the forward rate

**Question.** What is the value of protection against SOFR above 4.5%, and can
we reproduce the native Black caplet price using only curve and volatility queries?

Use the illustrative 24 February 2026 single-curve USD market, USD 1 million
notional, a one-year option expiry and three-month accrual. SOFR forwards use
simple Actual/360; the engine also uses Actual/360 expiry time here. Payment
is at accrual end. The Black model assumes a positive lognormal forward and
positive strike; this example is not a daily-observed compounded-in-arrears
SOFR cap. [Setup](../../README.md#install-and-run).

For forward $F$, strike $K$, volatility $\sigma$, expiry $t$, accrual $\tau$:
$$C=NP(0,T)\tau[F\Phi(d_1)-K\Phi(d_2)],\quad
d_1=\frac{\log(F/K)+\sigma^2t/2}{\sigma\sqrt t},\quad
d_2=d_1-\sigma\sqrt t.$$


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## A caplet and a strip of caplets

The caplet pays `notional × accrual × max(SOFR − strike, 0)` at the end of
one future accrual period. Fixing, accrual start, accrual end, and payment date
are separate terms even when some dates coincide. A `CapFloor` builds a strip
of these periods from a frequency. Every strike below is an absolute decimal
rate: `0.045` means 4.5%, not a relative moneyness adjustment. Relative-strike
interpretation is consumer-specific; this study deliberately avoids it.

The Python binding dispatches these products to the Black closed-form pricers.
The SOFR curve determines the forward and discount factor; the SOFR Black
surface provides volatility at the option expiry and strike. Attaching the
surface is necessary even though it was unnecessary for the swap and bond.

In [2]:
fixing = ref + "1Y"
payment = fixing + "3M"
caplet = qs.CapletFloorlet(
    identifier="USD-CAPLET-1Y",
    fixing_date=fixing,
    start_accrual_date=fixing,
    end_accrual_date=payment,
    payment_date=payment,
    strike=0.045,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    payoff_type=qs.CapletFloorletType.Caplet,
)
cap = qs.CapFloor(
    identifier="USD-CAP-1Y-3Y",
    start_date=fixing,
    maturity_date=ref + "3Y",
    strike=0.045,
    notional=1_000_000,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    cap_floor_type=qs.CapFloorType.Cap,
    frequency=qs.Frequency.Quarterly,
)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
)

with context:
    caplet_result = context.evaluate(
        qs.Trade(caplet, ref, notional=1_000_000),
        [qs.Request.Value, qs.Request.Sensitivities],
    )
    cap_result = context.evaluate(
        qs.Trade(cap, ref),
        [qs.Request.Value, qs.Request.Sensitivities],
    )

pd.DataFrame(
    [
        {"product": "single caplet", "pv_usd": caplet_result.price},
        {"product": "quarterly cap strip", "pv_usd": cap_result.price},
    ]
)

,product,pv_usd
0,single caplet,834.330964
1,quarterly cap strip,6067.398522


The caplet risk contains both interest-rate and volatility quote labels. A
volatility quote of `0.30` is a 30% Black volatility; a +1 volatility-point move
is `0.01`, whereas +1 bp on a rate is `0.0001`. Do not scale the whole table by
the same number and call it DV01. The next notebook separates these shocks.

In [3]:
caplet_result.sensitivities.sort_values(
    "value",
    key=lambda series: series.abs(),
    ascending=False,
).head(10)


,pillar,value
30,OIS_USD_SOFR_2Y,196958.793948
29,OIS_USD_SOFR_1Y,-95220.527119
7,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_S...,3888.965205
27,FixedRateDeposit_USD_SOFR_1D,-2.317296
25,CapletFloorlet_USD_SOFR_3M_7Y_Absolute_0.045_S...,0.000000
20,CapletFloorlet_USD_SOFR_3M_5Y_Absolute_0.055_S...,0.000000
21,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.035_S...,0.000000
22,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.045_S...,0.000000
23,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.055_S...,0.000000
24,CapletFloorlet_USD_SOFR_3M_7Y_Absolute_0.035_S...,0.000000


In [4]:
assert caplet_result.price > 0
assert cap_result.price > caplet_result.price
assert not caplet_result.sensitivities.empty
print("Both option values are positive; the cap strip contains multiple caplets.")

Both option values are positive; the cap strip contains multiple caplets.


## An independent Black price, put–call parity, and strip decomposition

Caplet minus floorlet must equal the discounted forward payoff
$NP(0,T)\tau(F-K)$. A cap is the sum of its constituent caplets, not one option
on an average rate. We construct every quarterly period explicitly below;
the sum checks scheduling as well as option arithmetic.


In [5]:
import math

def normal_cdf(value):
    return (1 + math.erf(value / math.sqrt(2))) / 2

with context:
    curve = context.curve(qs.MarketIndex.SOFR)
    surface = context.volatility_surface(qs.MarketIndex.SOFR)
    forward = curve.forward_rate(fixing, payment, qs.Compounding.Simple, qs.Frequency.Annual)
    discount = curve.discount_factor(payment)
    sigma = surface.volatility(fixing, 0.045)
    expiry_time = qs.DayCounter.Actual360.year_fraction(ref, fixing)
    accrual = qs.DayCounter.Actual360.year_fraction(fixing, payment)
    standard_deviation = sigma * math.sqrt(expiry_time)
    d1 = (math.log(forward / 0.045) + standard_deviation**2 / 2) / standard_deviation
    d2 = d1 - standard_deviation
    black_price = 1_000_000 * discount * accrual * (
        forward * normal_cdf(d1) - 0.045 * normal_cdf(d2)
    )
    floorlet = qs.CapletFloorlet(
        identifier="PARITY-FLOORLET", fixing_date=fixing,
        start_accrual_date=fixing, end_accrual_date=payment, payment_date=payment,
        strike=0.045, currency=qs.Currency.USD, market_index=qs.MarketIndex.SOFR,
        payoff_type=qs.CapletFloorletType.Floorlet,
    )
    floorlet_price = context.evaluate(
        qs.Trade(floorlet, ref, notional=1_000_000), [qs.Request.Value]
    ).price
    strip = []
    for period in range(8):
        start = fixing + qs.Period(3 * period, qs.TimeUnit.Months)
        end = fixing + qs.Period(3 * (period + 1), qs.TimeUnit.Months)
        component = qs.CapletFloorlet(
            identifier=f"CAPLET-{period + 1}", fixing_date=start,
            start_accrual_date=start, end_accrual_date=end, payment_date=end,
            strike=0.045, currency=qs.Currency.USD, market_index=qs.MarketIndex.SOFR,
            payoff_type=qs.CapletFloorletType.Caplet,
        )
        price = context.evaluate(qs.Trade(component, ref, notional=1_000_000), [qs.Request.Value]).price
        strip.append({"fixing_date": str(start), "payment_date": str(end), "pv_usd": price})

assert math.isclose(black_price, caplet_result.price, rel_tol=1e-10, abs_tol=1e-7)
assert math.isclose(
    caplet_result.price - floorlet_price,
    1_000_000 * discount * accrual * (forward - 0.045), abs_tol=1e-7,
)
assert math.isclose(sum(item["pv_usd"] for item in strip), cap_result.price, abs_tol=1e-6)
print(f"Forward {forward:.6%}; Black vol {sigma:.2%}; native/manual caplet USD {black_price:,.6f}.")
print(f"Eight caplets sum to USD {cap_result.price:,.6f}.")
pd.DataFrame(strip)


Forward 4.175676%; Black vol 29.00%; native/manual caplet USD 834.330964.
Eight caplets sum to USD 6,067.398522.


,fixing_date,payment_date,pv_usd
0,2027-02-24,2027-05-24,834.330964
1,2027-05-24,2027-08-24,898.368560
2,2027-08-24,2027-11-24,958.735768
3,2027-11-24,2028-02-24,1000.646019
4,2028-02-24,2028-05-24,571.899568
5,2028-05-24,2028-08-24,584.505071
6,2028-08-24,2028-11-24,603.429747
7,2028-11-24,2029-02-24,615.482826


## Risk units and conclusion

The direct formula and parity check validate the single-period payoff; the
strip sum validates the cap interpretation. Positive time value is not the same
as a positive forward payoff: protection can be valuable even when today's
forward is below strike. Rate derivatives require a `1e-4` bump for one bp;
Black-vol derivatives require `0.01` for one volatility point. Keep their
P&L units separate in the risk report.


In [6]:
risk = caplet_result.sensitivities.copy()
rate_labels = set(market.curves[0].to_dict()["quotes"])
vol_labels = set(market.volatility_surfaces[0].to_dict()["quotes"])
assert set(risk["pillar"]) <= rate_labels | vol_labels
risk["input_family"] = risk["pillar"].map(
    lambda label: "rate +1 bp" if label in rate_labels else "Black vol +1 point"
)
risk["pnl_usd"] = risk["value"] * risk["pillar"].map(
    lambda label: 1e-4 if label in rate_labels else 0.01
)
risk.loc[risk["pnl_usd"].abs() > 1e-8, ["pillar", "input_family", "pnl_usd"]]


,pillar,input_family,pnl_usd
7,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_S...,Black vol +1 point,38.889652
27,FixedRateDeposit_USD_SOFR_1D,rate +1 bp,-0.000232
29,OIS_USD_SOFR_1Y,rate +1 bp,-9.522053
30,OIS_USD_SOFR_2Y,rate +1 bp,19.695879


Useful cap/caplet requests are `Value` and `Sensitivities`. Black volatility
quote risk and rate quote risk use different bump sizes. Continue with
[curves and volatility](../../volatilitysurface/curves_and_volatility.ipynb)
for independent market shocks.